# Leonardo Challenge 2024

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import pandas as pd
import os
from transformers import ViTModel, BertModel, BertTokenizer
from transformers import ViTModel, GPT2Model, GPT2Tokenizer
from transformers import SwinModel

from tqdm import tqdm
from sklearn.model_selection import train_test_split

# Data Load

In [2]:
class MultimodalDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None, tokenizer=None, is_test=False):
        self.metadata = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        self.tokenizer = tokenizer or BertTokenizer.from_pretrained('bert-base-uncased')
        self.label_mapping = {label: idx for idx, label in enumerate(self.metadata['object'].unique())}
        self.is_test = is_test

    def __len__(self):
        return len(self.metadata)
    
    def __getitem__(self, idx):
        img_name = os.path.join(self.img_dir, self.metadata.iloc[idx, 0])
        image = Image.open(img_name).convert('RGB')
        object_name = self.metadata.iloc[idx, 1]
        #description = self.metadata.iloc[idx, 2]
        description = object_name + ": " + self.metadata.iloc[idx, 2]

        if self.transform:
            image = self.transform(image)

        object_label = self.label_mapping[object_name]
        
        encoded_text = self.tokenizer(description, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
        input_ids = encoded_text['input_ids'].squeeze(0)
        attention_mask = encoded_text['attention_mask'].squeeze(0)

        sample = {
            'image': image, 
            'object': object_label, 
            'input_ids': input_ids, 
            'attention_mask': attention_mask,
            'image_name': self.metadata.iloc[idx, 0]
        }
        
        if not self.is_test:
            sample['target'] = torch.tensor(self.metadata.iloc[idx, 3], dtype=torch.long)

        return sample

In [3]:
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.25, 0.25, 0.25])
    ])

    #tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token  # add padding

    # Load the full training dataset
    full_train_dataset = MultimodalDataset(csv_file='dataset/train.csv', img_dir='dataset/images/train', transform=transform, tokenizer=tokenizer)
    
    # Split the training dataset into train and validation sets
    train_size = int(0.9 * len(full_train_dataset))
    val_size = len(full_train_dataset) - train_size
    train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

    # Load the test dataset
    test_dataset = MultimodalDataset(csv_file='dataset/test.csv', img_dir='dataset/images/test', transform=transform, tokenizer=tokenizer, is_test=True)

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


# Model

In [4]:
class MultimodalClassifier(nn.Module):
    def __init__(self, num_classes, num_object_categories):
        super(MultimodalClassifier, self).__init__()
        self.image_encoder = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
        #self.text_encoder = BertModel.from_pretrained('bert-base-uncased')
        self.text_encoder = GPT2Model.from_pretrained('gpt2')

        
        self.image_projection = nn.Linear(self.image_encoder.config.hidden_size, 768)
        self.text_projection = nn.Linear(self.text_encoder.config.hidden_size, 768)
        #self.object_embedding = nn.Embedding(num_object_categories, 768)
        
        self.fusion = nn.Sequential(
            nn.Linear(768 * 2, 256),
#            nn.Linear(768 * 3, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, image, input_ids, attention_mask, object_label):
        image_features = self.image_projection(self.image_encoder(image).last_hidden_state[:, 0])
        text_features = self.text_projection(self.text_encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state)
        #object_features = self.object_embedding(object_label)

        # Mean
        text_features = torch.mean(text_features, dim=1)
        #image_features = torch.mean(image_features, dim=1)

        # Normalize
        image_features = F.normalize(image_features, dim=-1)
        text_features = F.normalize(text_features, dim=-1)
        #object_features = F.normalize(object_features, dim=-1)

        # Combine features
        combined_features = torch.cat([image_features, text_features], dim=1)
#        combined_features = torch.cat([image_features, text_features], dim=1)

        output = self.fusion(combined_features)
        return output

# Train

In [5]:
def train(model, train_loader, val_loader, optimizer, device, epochs):
    model.train()
    best_val_acc = 0
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for batch in progress_bar:
            images = batch['image'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            object_labels = batch['object'].to(device)
            targets = batch['target'].to(device)

            optimizer.zero_grad()
            outputs = model(images, input_ids, attention_mask, object_labels)
            loss = F.cross_entropy(outputs, targets)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            progress_bar.set_postfix({'loss': loss.item(), 'acc': 100. * correct / total})

        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {total_loss/len(train_loader):.4f}, Train Acc: {100. * correct / total:.2f}%")
        
        # Validation
        val_loss, val_acc = evaluate(model, val_loader, device)
        print(f"Validation Loss: {val_loss:.4f}, Validation Acc: {val_acc:.2f}%")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_multimodal_classifier.pth')
            print(f"New best model saved with validation accuracy: {best_val_acc:.2f}%")

In [6]:
def evaluate(model, data_loader, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            images = batch['image'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            object_labels = batch['object'].to(device)
            
            outputs = model(images, input_ids, attention_mask, object_labels)
            
            if 'target' in batch:
                targets = batch['target'].to(device)
                loss = F.cross_entropy(outputs, targets)
                total_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()

    if total > 0:
        avg_loss = total_loss / len(data_loader)
        accuracy = 100. * correct / total
        return avg_loss, accuracy
    else:
        return None, None

In [7]:

def prepare_submission(model, test_loader, device, output_file='submission.csv'):
    model.eval()
    all_image_names = []
    all_predictions = []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Generating predictions"):
            images = batch['image'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            object_labels = batch['object'].to(device)
            
            outputs = model(images, input_ids, attention_mask, object_labels)
            _, predicted = outputs.max(1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_image_names.extend(batch['image_name'])

    results_df = pd.DataFrame({
        'image_name': all_image_names,
        'target': all_predictions
    })

    results_df.to_csv(output_file, index=False)
    print(f"Submission file created: {output_file}")

In [8]:
# Main execution
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    num_classes = 4  # As specified in the problem description
    num_object_categories = len(full_train_dataset.label_mapping)

    model = MultimodalClassifier(num_classes, num_object_categories).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)

    num_epochs = 10
    train(model, train_loader, val_loader, optimizer, device, num_epochs)

    # Load the best model for final evaluation and submission
    model.load_state_dict(torch.load('best_multimodal_classifier.pth'))
    
    # Final validation
    val_loss, val_acc = evaluate(model, val_loader, device)
    print(f"Final Validation Loss: {val_loss:.4f}, Validation Acc: {val_acc:.2f}%")

    # Prepare submission
    prepare_submission(model, test_loader, device)

Using device: cuda


Epoch 1/10: 100%|██████████| 57/57 [00:34<00:00,  1.64it/s, loss=1.37, acc=30]  


Epoch 1/10, Train Loss: 1.3708, Train Acc: 30.00%


Evaluating: 100%|██████████| 7/7 [00:01<00:00,  3.89it/s]


Validation Loss: 1.3243, Validation Acc: 40.00%
New best model saved with validation accuracy: 40.00%


...

Validation Loss: 1.2260, Validation Acc: 66.25%


Evaluating: 100%|██████████| 7/7 [00:01<00:00,  3.95it/s]


Final Validation Loss: 0.8763, Validation Acc: 68.00%


Generating predictions: 100%|██████████| 16/16 [00:04<00:00,  3.57it/s]

Submission file created: submission.csv
